# Demo: Fitting ARIMA and SARIMAX with statsmodels


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Udacity brand colors
UB = {
    "Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"
}
UN = {
    "Black": "#0A0B0F", "Dark Gray": "#5A5C62",
    "Medium Gray": "#9C9FA8", "Gray": "#CECFD4",
    "Light Gray": "#F2F2F2", "White": "#FFFFFF"
}
US = {
    "Orange": "#EE7622", "Yellow": "#F9DC5C",
    "Red": "#9C0D22", "Green": "#23CE6B"
}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date")
df = df.asfreq("MS")

# Train on everything through 2023, test on 2024
train = df.loc[:"2023-12-01"]
print(f"Data: {len(df)} months")
print(f"Train: {len(train)} months, Test: {len(df) - len(train)} months")

## ACF/PACF


In [ ]:
axes[0].set_ylabel("Correlation", fontsize=16)
diffed = train["demand_mwh"].diff(12).dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(diffed, ax=axes[0], lags=36)
plot_pacf(diffed, ax=axes[1], lags=36)
axes[0].set_title("ACF (seasonally differenced)", fontsize=18, fontweight="bold")
axes[0].tick_params(axis="both", labelsize=14)
axes[1].set_title("PACF (seasonally differenced)", fontsize=18, fontweight="bold")
axes[1].tick_params(axis="both", labelsize=14)
plt.tight_layout()
plt.show()

## ARIMA(2,1,1) -- No Seasonality


In [ ]:
arima = SARIMAX(train["demand_mwh"], order=(2, 1, 1)).fit(disp=False)
arima_fc = arima.get_forecast(12)
arima_mean = arima_fc.predicted_mean
arima_ci = arima_fc.conf_int()
print(f"AIC: {arima.aic:.1f}")

## SARIMAX(1,1,1)(1,1,1,12) + Temperature


In [ ]:
sarimax = SARIMAX(
    train["demand_mwh"],
    exog=train["avg_temp_f"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12)
).fit(disp=False)

# For the forecast, we need future temperatures. Use seasonal averages from training data.
seasonal_temp = train.groupby(train.index.month)["avg_temp_f"].mean()
future_temp = pd.Series(
    [seasonal_temp[m] for m in range(1, 13)],
    index=pd.date_range("2024-01-01", periods=12, freq="MS")
)

sarimax_fc = sarimax.get_forecast(12, exog=future_temp.values)
sarimax_mean = sarimax_fc.predicted_mean
sarimax_ci = sarimax_fc.conf_int()
print(f"AIC: {sarimax.aic:.1f}")

## Comparing the Forecasts


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for ax, name, mean, ci, color in [
    (axes[0], "ARIMA(2,1,1)", arima_mean, arima_ci, UB["Brand Blue"]),
    (axes[1], "SARIMAX + temperature", sarimax_mean, sarimax_ci, US["Orange"]),
]:
    ax.plot(train.index, train["demand_mwh"].values, color=UN["Black"], label="Historical")
    ax.plot(mean.index, mean.values, color=color, linestyle="--", label="Forecast")
    ax.fill_between(mean.index, ci.iloc[:, 0], ci.iloc[:, 1], color=color, alpha=0.15, label="95% interval")
    ax.set_ylabel("Demand (MWh)", fontsize=16)
    ax.set_title(name, fontsize=18, fontweight="bold")
    ax.tick_params(axis="both", labelsize=14)
    ax.legend(loc="upper left")

plt.tight_layout()
plt.show()